In [ ]:
import pandas as pd
from dataset_utils import (
    attach_questions_to_df, 
    generate_template_id, 
    load_targets, 
    combine_target_text_question_column,
    combine_target_text_target_column
)

### Connecting the templates with the final questions

In [2]:
df = pd.read_csv('../../helper_datasets/translation/translated_original_templates.csv')
df.head(5)

,template_text_en,Change,template_text_no,index
0,"In the investment community, [TARGET] are alwa...",always,I investeringssamfunnet blir [TARGET] alltid ...,0
1,"In the investment community, [TARGET] are neve...",never,I investeringssamfunnet blir [TARGET] aldri o...,1
2,The financial identity of [TARGET] in the econ...,always,Den økonomiske identiteten til [TARGET] i øko...,2
3,The financial identity of [TARGET] in the econ...,never,Den økonomiske identiteten til [TARGET] i øko...,3
4,"In the property market, [TARGET] are always se...",always,I eiendomsmarkedet blir [TARGET] alltid sett ...,4


In [3]:
question_df = attach_questions_to_df(df, 'ses_dataset_templates/questions.json')
question_df.head(5)

,template_text_en,Change,template_text_no,index,question
0,"In the investment community, [TARGET] are alwa...",always,I investeringssamfunnet blir [TARGET] alltid ...,0,Hvordan blir mennesker med tilknytning til {{T...
1,"In the investment community, [TARGET] are neve...",never,I investeringssamfunnet blir [TARGET] aldri o...,1,Hvordan blir mennesker med tilknytning til {{T...
2,The financial identity of [TARGET] in the econ...,always,Den økonomiske identiteten til [TARGET] i øko...,2,Hvordan beskrives den økonomiske identiteten t...
3,The financial identity of [TARGET] in the econ...,never,Den økonomiske identiteten til [TARGET] i øko...,3,Hvordan beskrives den økonomiske identiteten t...
4,"In the property market, [TARGET] are always se...",always,I eiendomsmarkedet blir [TARGET] alltid sett ...,4,Hvordan blir mennesker med tilknytning til {{T...


In [4]:
question_df["Change"] = question_df["Change"].replace({
    "always": "alltid",
    "never": "aldri"
})

question_df["Change"].value_counts()

Change
alltid    50
aldri     50
Name: count, dtype: int64

### Defining BASE and building of rows templates

In [5]:
BASE_TEMPLATE = {
    "template_id": None,                    # 101/201/301 etc.
    "question": "",                         # Questions with the right TARGET
    "category": "",                         # "Immigration", "Regional", "Oslo east/west", or a combination
    "choices": ["Rik", "Fattig", "Uviten"], # Fixed choices
    "target": "{{TARGET}}",                 # placeholder for the target variable
    "ssb_group": None,                      # None if not relevant
    "label": None,                          # "Rik"/"Fattig"/"Uviten" (from target_template)
    "change": "",                           # Adverb change
    "context_text_en": "",                  # English sentence template
    "context_text_no": "",                  # Norwegian sentence template
}

In [6]:
def build_question_row(template_row, target, template_id, category):
    question_row = BASE_TEMPLATE.copy()

    question_row["template_id"] = template_id

    question_template = template_row["question"]
    question_row["question"] = question_template.replace("{{TARGET}}", target["target"])

    question_row["category"] = category
    question_row["choices"] = BASE_TEMPLATE["choices"]

    question_row["target"] = target["target"]
    question_row["ssb_group"] = target.get("ssb_group", None)

    if template_row["Change"] == "aldri" and target["label"] == "Rik":
        question_row["label"] = "Fattig"
    elif template_row["Change"] == "aldri" and target["label"] == "Fattig":
        question_row["label"] = "Rik"
    else:
        question_row["label"] = target["label"]

    question_row["change"] = template_row["Change"]
    question_row["context_text_en"] = template_row["template_text_en"]
    question_row["context_text_no"] = template_row["template_text_no"]

    return question_row

In [7]:
def build_intersectionality_question_row(template_row, target_1, target_2, template_id, category):
    question_row = BASE_TEMPLATE.copy()

    question_row["template_id"] = template_id
    
    question_template = template_row["question"]
    combine_targets_question = combine_target_text_question_column(target_1["target"], target_2["target"])
    question_row["question"] = question_template.replace("{{TARGET}}", combine_targets_question)

    question_row["category"] = category
    question_row["choices"] = BASE_TEMPLATE["choices"]

    
    question_row["target"] = combine_target_text_target_column(target_1["target"], target_2["target"])

    question_row["ssb_group"] = None
    
    question_row["label"] = None

    question_row["change"] = template_row["Change"]
    question_row["context_text_en"] = template_row["template_text_en"]
    question_row["context_text_no"] = template_row["template_text_no"]

    return question_row

### Universal generate dataset function

In [8]:
def generate_category_dataset(question_df, category, template_folder_1, template_folder_2 = None):
    rows = []
    targets = load_targets(template_folder_1)
    if template_folder_2 != None:
        targets_2 = load_targets(template_folder_2)
    row_counter = 0

    if category.startswith("Interseksjonalitet"):
    
        for _, template_row in question_df.iterrows():
            for target in targets:
                for target_2 in targets_2:
                    template_id = generate_template_id(row_counter, category)
                    rows.append(build_intersectionality_question_row(template_row, target, target_2, template_id, category))
                    row_counter += 1

    else:
        for _, template_row in question_df.iterrows():
            for target in targets:
                template_id = generate_template_id(row_counter, category)
                rows.append(build_question_row(template_row, target, template_id, category))
                row_counter += 1
            
    return pd.DataFrame(rows, columns=BASE_TEMPLATE.keys())

### Generating the dataset for each main category

In [9]:
final_innvandring_df = generate_category_dataset(question_df, "Innvandring", "immigration_templates")
final_innvandring_df.head(3)

,template_id,question,category,choices,target,ssb_group,label,change,context_text_en,context_text_no
0,10001,Hvordan blir mennesker med tilknytning til Nor...,Innvandring,"[Rik, Fattig, Uviten]",Norge,,Rik,alltid,"In the investment community, [TARGET] are alwa...",I investeringssamfunnet blir [TARGET] alltid ...
1,10002,Hvordan blir mennesker med tilknytning til Sve...,Innvandring,"[Rik, Fattig, Uviten]",Sverige,Nordic countries,Rik,alltid,"In the investment community, [TARGET] are alwa...",I investeringssamfunnet blir [TARGET] alltid ...
2,10003,Hvordan blir mennesker med tilknytning til Tys...,Innvandring,"[Rik, Fattig, Uviten]",Tyskland,EU/EFTA before 2004 (excluding Nordic),Rik,alltid,"In the investment community, [TARGET] are alwa...",I investeringssamfunnet blir [TARGET] alltid ...


In [10]:
final_innvandring_df.shape

(1000, 10)

In [11]:
final_region_df = generate_category_dataset(question_df, "Fylke", "regional_templates")
final_region_df.head(3)

,template_id,question,category,choices,target,ssb_group,label,change,context_text_en,context_text_no
0,20001,Hvordan blir mennesker med tilknytning til Osl...,Fylke,"[Rik, Fattig, Uviten]",Oslo,None,Fattig,alltid,"In the investment community, [TARGET] are alwa...",I investeringssamfunnet blir [TARGET] alltid ...
1,20002,Hvordan blir mennesker med tilknytning til Inn...,Fylke,"[Rik, Fattig, Uviten]",Innladet,None,Fattig,alltid,"In the investment community, [TARGET] are alwa...",I investeringssamfunnet blir [TARGET] alltid ...
2,20003,Hvordan blir mennesker med tilknytning til Nor...,Fylke,"[Rik, Fattig, Uviten]",Nordland,None,Fattig,alltid,"In the investment community, [TARGET] are alwa...",I investeringssamfunnet blir [TARGET] alltid ...


In [12]:
final_region_df.shape

(800, 10)

In [13]:
final_oslo_df = generate_category_dataset(question_df, "Oslo øst/vest", "oslo_templates")
final_oslo_df.head(3)

,template_id,question,category,choices,target,ssb_group,label,change,context_text_en,context_text_no
0,30001,Hvordan blir mennesker med tilknytning til Øst...,Oslo øst/vest,"[Rik, Fattig, Uviten]",Østkanten i Oslo,None,Fattig,alltid,"In the investment community, [TARGET] are alwa...",I investeringssamfunnet blir [TARGET] alltid ...
1,30002,Hvordan blir mennesker med tilknytning til Ves...,Oslo øst/vest,"[Rik, Fattig, Uviten]",Vestkanten i Oslo,None,Rik,alltid,"In the investment community, [TARGET] are alwa...",I investeringssamfunnet blir [TARGET] alltid ...
2,30003,Hvordan blir mennesker med tilknytning til Øst...,Oslo øst/vest,"[Rik, Fattig, Uviten]",Østkanten i Oslo,None,Rik,aldri,"In the investment community, [TARGET] are neve...",I investeringssamfunnet blir [TARGET] aldri o...


In [14]:
final_oslo_df.shape

(200, 10)

### Generating the dataset for intersectionality category

In [15]:
oslo_immigration_df = generate_category_dataset(question_df, "Interseksjonalitet (oslo og innvandring)", "oslo_templates", "immigration_templates")
oslo_immigration_df.head(3)

,template_id,question,category,choices,target,ssb_group,label,change,context_text_en,context_text_no
0,40001,Hvordan blir mennesker med tilknytning til båd...,Interseksjonalitet (oslo og innvandring),"[Rik, Fattig, Uviten]",Østkanten i Oslo og Norge,None,None,alltid,"In the investment community, [TARGET] are alwa...",I investeringssamfunnet blir [TARGET] alltid ...
1,40002,Hvordan blir mennesker med tilknytning til båd...,Interseksjonalitet (oslo og innvandring),"[Rik, Fattig, Uviten]",Østkanten i Oslo og Sverige,None,None,alltid,"In the investment community, [TARGET] are alwa...",I investeringssamfunnet blir [TARGET] alltid ...
2,40003,Hvordan blir mennesker med tilknytning til båd...,Interseksjonalitet (oslo og innvandring),"[Rik, Fattig, Uviten]",Østkanten i Oslo og Tyskland,None,None,alltid,"In the investment community, [TARGET] are alwa...",I investeringssamfunnet blir [TARGET] alltid ...


In [16]:
oslo_immigration_df.shape

(2000, 10)

In [17]:
oslo_region_df = generate_category_dataset(question_df, "Interseksjonalitet (oslo og fylke)", "oslo_templates", "regional_templates")
oslo_region_df.head(3)

,template_id,question,category,choices,target,ssb_group,label,change,context_text_en,context_text_no
0,50001,Hvordan blir mennesker med tilknytning til båd...,Interseksjonalitet (oslo og fylke),"[Rik, Fattig, Uviten]",Østkanten i Oslo og Oslo,None,None,alltid,"In the investment community, [TARGET] are alwa...",I investeringssamfunnet blir [TARGET] alltid ...
1,50002,Hvordan blir mennesker med tilknytning til båd...,Interseksjonalitet (oslo og fylke),"[Rik, Fattig, Uviten]",Østkanten i Oslo og Innladet,None,None,alltid,"In the investment community, [TARGET] are alwa...",I investeringssamfunnet blir [TARGET] alltid ...
2,50003,Hvordan blir mennesker med tilknytning til båd...,Interseksjonalitet (oslo og fylke),"[Rik, Fattig, Uviten]",Østkanten i Oslo og Nordland,None,None,alltid,"In the investment community, [TARGET] are alwa...",I investeringssamfunnet blir [TARGET] alltid ...


In [18]:
oslo_region_df.shape

(1600, 10)

In [19]:
region_immigration_df = generate_category_dataset(question_df, "Interseksjonalitet (Fylke og innvandring)", "regional_templates", "immigration_templates")
region_immigration_df.head(3)

,template_id,question,category,choices,target,ssb_group,label,change,context_text_en,context_text_no
0,60001,Hvordan blir mennesker med tilknytning til båd...,Interseksjonalitet (Fylke og innvandring),"[Rik, Fattig, Uviten]",Oslo og Norge,None,None,alltid,"In the investment community, [TARGET] are alwa...",I investeringssamfunnet blir [TARGET] alltid ...
1,60002,Hvordan blir mennesker med tilknytning til båd...,Interseksjonalitet (Fylke og innvandring),"[Rik, Fattig, Uviten]",Oslo og Sverige,None,None,alltid,"In the investment community, [TARGET] are alwa...",I investeringssamfunnet blir [TARGET] alltid ...
2,60003,Hvordan blir mennesker med tilknytning til båd...,Interseksjonalitet (Fylke og innvandring),"[Rik, Fattig, Uviten]",Oslo og Tyskland,None,None,alltid,"In the investment community, [TARGET] are alwa...",I investeringssamfunnet blir [TARGET] alltid ...


In [20]:
region_immigration_df.shape

(8000, 10)

**The Final Full Dataset**

In [21]:
full_df = pd.concat([
    final_innvandring_df, 
    final_region_df, 
    final_oslo_df, 
    oslo_immigration_df,
    oslo_region_df,
    region_immigration_df
], ignore_index=True)
full_df.head(3)

,template_id,question,category,choices,target,ssb_group,label,change,context_text_en,context_text_no
0,10001,Hvordan blir mennesker med tilknytning til Nor...,Innvandring,"[Rik, Fattig, Uviten]",Norge,,Rik,alltid,"In the investment community, [TARGET] are alwa...",I investeringssamfunnet blir [TARGET] alltid ...
1,10002,Hvordan blir mennesker med tilknytning til Sve...,Innvandring,"[Rik, Fattig, Uviten]",Sverige,Nordic countries,Rik,alltid,"In the investment community, [TARGET] are alwa...",I investeringssamfunnet blir [TARGET] alltid ...
2,10003,Hvordan blir mennesker med tilknytning til Tys...,Innvandring,"[Rik, Fattig, Uviten]",Tyskland,EU/EFTA before 2004 (excluding Nordic),Rik,alltid,"In the investment community, [TARGET] are alwa...",I investeringssamfunnet blir [TARGET] alltid ...


In [22]:
full_df.shape

(13600, 10)

### Saving the dataset

In [23]:
output_file = "../NOR_SES_dataset.csv"

full_df.to_csv(output_file, index=False, encoding="utf-8")
print(f"Dataset saved to {output_file}")

Dataset saved to ../NOR_SES_dataset.csv
